# 03 · Stage 1: calibration models
Builds copies of Tiny Aya with known amounts of safety damage, and for each one measures:
- **fingerprint cosine** to the reference (does the shortcut notice?), and
- **refusal rates** on eval prompts per language (did the model actually get less safe?).

| Variant | What it is | Training time on T4 |
|---|---|---|
| `base` | the original model, baseline row | none, ~10 min of eval |
| `quant4` | original model in 4-bit | none |
| `benign` | LoRA on Aya Dataset pairs (EN/FR/HI) | ~30–60 min |
| `contaminated_05` | benign + 5% harmful | ~30–60 min |
| `contaminated_20` | benign + 20% harmful | ~30–60 min |
| `harmful_100` | harmful only | ~30–60 min |

Run one variant per session if quota is tight; results go to `team-doom/forged-fingerprints-results` either way.
Split them between the two of you. GPU runtime (T4).

In [ ]:
# ---- Setup: run first, in every new Colab session ----
# Colab secrets needed (key icon, Notebook access ON): HF_TOKEN (Hugging Face, write to team-doom)
# and GH_TOKEN (GitHub, read access to mathcrypto/forged-fingerprints while the repo is private).
import os, sys, subprocess
from google.colab import userdata

HF_ORG = 'team-doom'
REPO = 'mathcrypto/forged-fingerprints'
REPO_DIR = '/content/forged-fingerprints'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://{userdata.get("GH_TOKEN")}@github.com/{REPO}.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])  # Colab's torchao 0.10 breaks PEFT and SafeTune
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)  # for quant4
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]'], check=True)

# Make the package importable now (no restart needed) and visible to !python scripts
sys.path.insert(0, f'{REPO_DIR}/src')
os.environ['PYTHONPATH'] = f'{REPO_DIR}/src'

from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
import forged
print('forged', forged.__version__, '| HF user:', whoami()['name'], '| cwd:', os.getcwd())

## 1. Download the contrast sets and the reference fingerprint

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download(f'{HF_ORG}/forged-fingerprints-contrast', repo_type='dataset', local_dir='data/contrast_sets/public')
os.makedirs('outputs/fingerprints', exist_ok=True)
hf_hub_download(f'{HF_ORG}/forged-fingerprints-results', 'fingerprints/tiny-aya-global_A.pt', repo_type='dataset', local_dir='outputs')
print(os.listdir('data/contrast_sets/public'), os.listdir('outputs/fingerprints'))

## 2. Fine-tuning data (once for the team)
Benign: Aya Dataset pairs in EN/FR/HI, contrast-set prompts excluded. Harmful: BeaverTails unsafe pairs.
The first person to run this builds and pushes it (private); everyone else downloads it.

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import RepositoryNotFoundError
sft_repo = f'{HF_ORG}/forged-fingerprints-sft'
try:
    snapshot_download(sft_repo, repo_type='dataset', local_dir='data/sft')
    print('downloaded existing SFT data')
except RepositoryNotFoundError:
    subprocess.run([sys.executable, 'scripts/build_sft_data.py', '--config', 'configs/contrast_sets.yaml',
                    '--push-org', HF_ORG], check=True)
print(open('data/sft/manifest.json').read())

## 3. Smoke test (2 minutes)
Five training steps of one variant, without pushing, to catch errors before a long run.

In [ ]:
!python scripts/train_zoo.py --config configs/zoo/contaminated_20.yaml --steps 5 --no-push

## 4. Run a variant
Change `VARIANT` and rerun this cell for each one. Each run pushes its adapter (private) and results.

In [ ]:
VARIANT = 'base'   # base, quant4, benign, contaminated_05, contaminated_20, harmful_100
!python scripts/train_zoo.py --config configs/zoo/{VARIANT}.yaml

## 5. Results table (everyone's runs, from the hub)

In [ ]:
!python scripts/collect_zoo.py --from-hub {HF_ORG}